# T1.2 / T1.3 — Pré-processamento e modelo baseline (XGBoost)

Constrói as 9 features do contrato, faz o split temporal, treina o baseline e
avalia as métricas iniciais (D2).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from data.loader import load_paysim
from data.preprocess import FEATURES, TYPE_CODES, prepare_dataset
from ml.train import train_baseline

sns.set_theme(style="whitegrid", context="notebook")

## 1. Features

O mapa `TYPE_CODES` é fixo e faz parte do artefato: treino e inferência usam
exatamente a mesma codificação.

In [ ]:
print(TYPE_CODES)
print(FEATURES)

## 2. Split temporal

O teste é sempre o futuro do treino — nunca um split aleatório.

In [ ]:
df = load_paysim(rows=60_000, seed=42)
split = prepare_dataset(df, train_ratio=0.7)
split.summary()

In [ ]:
split.X_train.head()

## 3. Treino do baseline

O artefato é salvo em `MODEL_PATH` (container) ou `backend/models` (local).

In [ ]:
result = train_baseline(df, rows=len(df), version="v1.1.0", save=True)
print("versão:", result.version)
print("artefato:", result.model_path)
pd.Series(result.test_metrics).to_frame("valor")

## 4. Importância das features

In [ ]:
importance = pd.DataFrame(result.feature_importance)
importance

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=importance, x="importance", y="feature", ax=ax, color="#2563eb")
ax.set_title("Importância das features — baseline")
ax.set_xlabel("importância normalizada")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 5. Matriz de confusão no teste

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

scores = result.model.predict_proba(split.X_test)
predicted = (scores >= result.model.threshold).astype(int)
matrix = confusion_matrix(split.y_test.astype(int), predicted)

display = ConfusionMatrixDisplay(matrix, display_labels=["legítima", "fraude"])
display.plot(cmap="Blues", values_format="d")
plt.title(f"Baseline {result.version}")
plt.show()

## 6. Leitura

- O baseline aprende bem o padrão histórico (esvaziamento da origem).
- As métricas de teste são a referência do E1 e o ponto de partida do E2.
- O artefato versionado é consumido pela API (`ml.predict.FraudPredictor`).